<a href="https://colab.research.google.com/github/Aranzazu21/Hands-on-1-2/blob/master/Hands_on_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Perceptrón Multicapa (MLP) en Datos Bancarios Reales**
### **Tutorial basado en la implementación de Awhan Mohanty**

* **Jorge Limon Aranzazu**
* **220111773**

---
## **Descripción de la librería Scikit-Learn para MLPs**

Scikit-Learn (`sklearn`) es la biblioteca más robusta para Machine Learning en Python. Su implementación de redes neuronales se centra en el **Perceptrón Multicapa (MLP)**.

### ** Clases y funciones empleadas**
* **`MLPClassifier`**: Es el estimador principal para problemas de clasificación. Implementa el algoritmo de **Backpropagation** para entrenar la red.
* **`LabelEncoder`**: Se utiliza para normalizar etiquetas de texto y convertirlas en números enteros.
* **`StandardScaler`**: Realiza la estandarización de características eliminando la media y escalando a la varianza unitaria.
* **`train_test_split`**: Función para segmentar los datos en subconjuntos de entrenamiento y validación.

### ** Parámetros principales**
* **`hidden_layer_sizes`**: Tupla que representa el número de neuronas en cada capa oculta. Ejemplo: `(13, 10, 2)`.
* **`max_iter`**: El número máximo de épocas que el optimizador utilizará antes de detenerse.
* **`activation`**: Función para las capas ocultas (ej. 'relu', 'logistic').
* **`random_state`**: Garantiza que el entrenamiento sea reproducible.

## **Pipeline del Proyecto**

### **Descripción del problema de clasificación**
El objetivo es predecir un resultado **binario**: ¿Se suscribirá el cliente a un depósito a plazo fijo? (Sí/No). Este es un problema de clasificación supervisada basado en campañas de marketing telefónico de una institución bancaria portuguesa.

In [ ]:
import pandas as pd
import numpy as np
import os

# Librerías de Scikit-Learn (Matemáticas y Matrices)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ruta = "/content/sample_data/bank-additional-full.csv"

try:
    df = pd.read_csv(ruta, sep=';')
    print("¡Archivo cargado exitosamente!")
    print(f"La matriz original tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
    # Mostramos los primeros datos con print para evitar errores de Parseo
    print("\nVista previa de la matriz de datos:")
    print(df.head(3))
except Exception as e:
    print(f"Error al cargar: {e}")

¡Archivo cargado exitosamente!
La matriz original tiene 41188 filas y 21 columnas.

Vista previa de la matriz de datos:
   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx  euribor3m  nr.employed   y  
0          93.994          -36.4      4.857       5191.0  no  
1          93.994          -36.4      4.857       5191.0  no  
2          93.994          -36.4      4.857       5191.0  no 

### **Exploración y preprocesamiento del Dataset**

#### **Describir las variables: tipos e interpretación**
El dataset contiene variables de distintos tipos:
* **Demográficas:** `age` (numérica), `job` (tipo de trabajo), `marital` (estado civil).
* **Historial:** `default` (crédito en mora), `housing` (préstamo de vivienda).
* **Sociales:** `education` (nivel de estudios).
* **Campañas:** `contact` (medio de contacto), `duration` (segundos de la llamada).

#### **Variables independientes y dependiente**
* **Independientes ($X$):** Atributos del cliente (edad, educación, préstamos, etc.).
* **Dependiente ($y$):** La columna `y` (renombrada como `subscribed`), que es el objetivo binario.

#### **Variables eliminadas y ¿por qué?**
Se eliminan columnas como `day_of_week`, `month` y `default`.
* **Razón:** Se consideran "ruido" o variables menos significativas para la clasificación financiera según el análisis de Mohanty, buscando un modelo más ligero y enfocado en el perfil del cliente.

In [ ]:

# PREPROCESAMIENTO: CODIFICACIÓN Y LIMPIEZA

LE = LabelEncoder()

# Transformación de categorías a códigos numéricos
df['job_code'] = LE.fit_transform(df['job'])
df['marital_code'] = LE.fit_transform(df['marital'])
df['education_code'] = LE.fit_transform(df['education'])
df['housing_code'] = LE.fit_transform(df['housing'])
df['loan_code'] = LE.fit_transform(df['loan'])
df['contact_code'] = LE.fit_transform(df['contact'])
df['poutcome_code'] = LE.fit_transform(df['poutcome'])
df['subscribed'] = LE.fit_transform(df['y'])

# Eliminación de columnas redundantes y no deseadas
df = df.drop(['job','marital','education','housing','loan','contact','poutcome','y','day_of_week','month','default'], axis=1)

print("Columnas actuales en el dataset:")
print(df.columns)

Columnas actuales en el dataset:
Index(['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed',
       'job_code', 'marital_code', 'education_code', 'housing_code',
       'loan_code', 'contact_code', 'poutcome_code', 'subscribed'],
      dtype='object')


#### ** Estandarización: ¿Por qué y cómo?**
**¿Por qué?** En datos bancarios, la edad (ej. 45) y los códigos binarios (0 o 1) tienen escalas muy diferentes. El MLP es sensible a esto.
**¿Cómo?** Se usa `StandardScaler` para centrar los datos en **media 0** y **desviación 1**, permitiendo que el optimizador encuentre los pesos óptimos más rápido.

In [ ]:

# DIVISIÓN Y ESCALADO
X = df.drop('subscribed', axis=1)
y = df['subscribed']

# División del dataset (Mohanty usa por defecto 75% entrenamiento, 25% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Aplicar estandarización
scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

### **Model Selection**
La arquitectura propuesta por Mohanty para este problema bancario es:
* **Capa Entrada:** 13 neuronas (una por cada variable de entrada).
* **Capa Oculta:** 10 neuronas.
* **Capa Salida:** 2 neuronas (para la clasificación binaria).
* **Arquitectura:** `(13, 10, 2)`.

In [ ]:

# ENTRENAMIENTO (MODEL TRAINING)

# Definición del MLP con la arquitectura (13, 10, 2)
mlp = MLPClassifier(hidden_layer_sizes=(13, 10, 2), max_iter=1000, random_state=42)

# Entrenamiento mediante Backpropagation
mlp.fit(X_train, y_train)

print("El modelo ha sido entrenado exitosamente.")

El modelo ha sido entrenado exitosamente.


### **Model Prediction**
Creamos una función que reciba el perfil de un cliente y nos devuelva si es propenso a suscribirse o no, aplicando previamente el escalado.

In [ ]:
# Función de Predicción
def predecir_cliente(medidas_cliente):
    # El dato de entrada debe ser escalado
    entrada = scaler.transform([medidas_cliente])
    pred = mlp.predict(entrada)
    return "Suscrito (YES)" if pred[0] == 1 else "No Suscrito (NO)"

# Ejemplo con el primer registro del dataset
ejemplo = X.iloc[0].tolist()
print(f"Resultado de la predicción: {predecir_cliente(ejemplo)}")

Resultado de la predicción: No Suscrito (NO)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


### ** Model Evaluation**
Para validar la robustez en datos financieros, utilizamos la **Matriz de Confusión** y el **Accuracy Score** (Exactitud).

In [ ]:
# 1. Usamos el método .predict() del objeto mlp sobre los datos de prueba
predictions = mlp.predict(X_test)

print("--- MATRIZ DE CONFUSIÓN ---")
# Comparamos la realidad (y_test) contra lo que predijo la red (predictions)
print(confusion_matrix(y_test, predictions))

# 2. Calculamos la exactitud comparando y_test con las predicciones
accuracy = accuracy_score(y_test, predictions)
print(f"\nExactitud del modelo (Accuracy): {accuracy*100:.2f}%")

print("\n--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_test, predictions))

--- MATRIZ DE CONFUSIÓN ---
[[8803  341]
 [ 546  607]]

Exactitud del modelo (Accuracy): 91.39%

--- REPORTE DE CLASIFICACIÓN ---
              precision    recall  f1-score   support

           0       0.94      0.96      0.95      9144
           1       0.64      0.53      0.58      1153

    accuracy                           0.91     10297
   macro avg       0.79      0.74      0.76     10297
weighted avg       0.91      0.91      0.91     10297

